<a href="https://colab.research.google.com/github/sandeepvaid/AI_Advance_Topics/blob/main/embeding_vector_db_semantic_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Local Inference on GPU
Model page: https://huggingface.co/sentence-transformers/all-MiniLM-L12-v2

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/sentence-transformers/all-MiniLM-L12-v2)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L12-v2")

sentences = [
    "That is a happy person",
    "That is a happy dog",
    "That is a very happy person",
    "Today is a sunny day"
]
embeddings = model.encode(sentences)

similarities = model.similarity(embeddings, embeddings)
print(similarities.shape)
# [4, 4]

## Remote Inference via Inference Providers
Ensure you have a valid **HF_TOKEN** set in your environment. You can get your token from [your settings page](https://huggingface.co/settings/tokens). Note: running this may incur charges above the free tier.
The following Python example shows how to run the model remotely on HF Inference Providers, automatically selecting an available inference provider for you.
For more information on how to use the Inference Providers, please refer to our [documentation and guides](https://huggingface.co/docs/inference-providers/en/index).

In [ ]:
import os
os.environ['HF_TOKEN'] = 'YOUR_TOKEN_HERE'

In [ ]:
import os
from huggingface_hub import InferenceClient

client = InferenceClient(
    provider="auto",
    api_key=os.environ["HF_TOKEN"],
)

result = client.sentence_similarity(
    {
    "source_sentence": "That is a happy person",
    "sentences": [
        "That is a happy dog",
        "That is a very happy person",
        "Today is a sunny day"
    ]
},
    model="sentence-transformers/all-MiniLM-L12-v2",
)

In [ ]:
import requests
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# 🔹 Step 1: Fetch and clean webpage text
def fetch_text_from_url(url):
    headers = {
        "User-Agent": "MyApp/1.0 (contact: your_email@example.com)"
    }

    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    for script in soup(["script", "style"]):
        script.extract()

    text = soup.get_text(separator=" ")
    return " ".join(text.split())

# 🔹 Step 2: Split text into chunks
def chunk_text(text, chunk_size=200):
    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)

    return chunks


# 🔹 Step 3: Build FAISS index
def build_index(chunks, model):
    embeddings = model.encode(chunks)
    embeddings_np = np.array(embeddings).astype('float32')

    index = faiss.IndexFlatL2(embeddings_np.shape[1])
    index.add(embeddings_np)

    return index, embeddings_np


# 🔹 Step 4: Query function
def search(query, model, index, chunks, k=3):
    query_embedding = model.encode([query])
    D, I = index.search(np.array(query_embedding).astype('float32'), k)

    results = []
    for idx in I[0]:
        results.append(chunks[idx])

    return results


# 🔥 MAIN FLOW

url = "https://en.wikipedia.org/wiki/Artificial_intelligence"  # 👈 replace with your URL

# Load model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Fetch content
text = fetch_text_from_url(url)

# Chunk it
chunks = chunk_text(text)

# Build index
index, embeddings = build_index(chunks, model)

# Query
query = "What is AI used for?"
results = search(query, model, index, chunks)

print("\n🔍 Top Results:\n")
for i, res in enumerate(results):
    print(f"{i+1}. {res[:200]}...\n")